# Writing metrics

<img align="left" src = https://project.lsst.org/sites/default/files/Rubin-O-Logo_0.png width=250 style="padding: 10px"> 
<b>Writing metrics</b> <br>
Contact author: Eric Neilsen. <br>
Questions welcome at <a href="https://community.lsst.org/c/sci/survey-strategy">community.lsst.org/c/sci/survey-strategy</a> and the <a href="https://lsstc.slack.com/archives/C2LTWTP5J">#sims_operations</a> slack channel.<br>
Find additional MAF documentation and resources at <a href="https://rubin-sim.lsst.io">rubin-sim.lsst.io</a>. <br>

This notebook gives instructions on how to write `MAF` metrics, the most common code development task for science groups.

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-04
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/tutorial

## 1. Notebook preparation

### 1.1 Developer aids

The following is a development style aid; only uncomment if developing the notebook:

In [ ]:
# %load_ext lab_black
# %load_ext pycodestyle_magic
# %flake8_on --ignore E501,W505

### 1.2 Import the required python modules

In [ ]:
from rubin_sim import maf
# from rubin_scheduler.data import get_baseline

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

### 1.3 Get example data to work with

To test a metric, we will need to use a sample database. Let's use the sample baseline database installed as part of the `rubin_sim` installation process:

In [ ]:
opsim_db_fname = get_baseline()
opsim_db_fname

We will also need an output directory. If you have one you want to use, set it here:

In [ ]:
# output_dir = '.'

Otherwise, create a temporary directory that will automatically be cleaned up when this notebook shuts down:

In [ ]:
if "output_dir" not in locals():
    from tempfile import TemporaryDirectory
    import os

    output_dir_itself = TemporaryDirectory(prefix="02_writing_metrics_", dir=os.getcwd())
    output_dir = output_dir_itself.name

output_dir

## 2. Understanding how metrics are called

To understand how to write metrics, it helps to first know a little bit about slicers.

`MAF` slicers categorize visits in the `opsim` database into groups called "slices." Slices have a "many to many" relationship with visits: a given slice may contain zero or more visits, and a given visit may fall into zero, one, or many slices. For example, the `HealpixSlicer` slices the sky according to [healpixels](https://healpix.jpl.nasa.gov/). Because the sky in any given healpixel can be covered any number of times (including zero), the slice corresponding to a healpixel may contain zero, one, or many visits. A given visit may cover sky in multiple healpixels, and so may be present in multiple slices.

**Slicers themselves are python iterators that return tuples with indexes into the array of visits, and slice points (parameters for the slice)**:

MAF metric objects must all include (among other things): 
- a `col_name_array` member, a `numpy.array` of strings with the names of **database columns** and any **stackers** that the metric requires. The `__init__` method of the `rubin_sim.maf.metrics.BaseMetric` includes a `col` named argument which is normally used to set the `col_name_array`. When there is only one element in `col_name_array`, the `colname` member is set with that column name.
- a `run` method that takes two arguments:
  - `data_slice`, a `numpy.recarray` with one record per visit in a slice, and elements corresponding to (at least) all columns listed in `col_name_array`; and
  - `slice_point`, a python object (usually a dictionary) defining the parameters of a slice.
  and **returns an object that represents the value of the metric for that slice point**.

When MAF metrics are computed by calling the `run_all` method of an instance of `rubin_sim.maf.metricBundles.MetricBundleGroup`, the `MetricBundleGroup`:
1. queries the database for any columns required (as specified by the `col_name_array` member of the metric object),
2. computes values for any stackers listen in the metric's `col_name_array` member, and
3. iterates over each slice, calling the metric's `run` method for every slice, and filling the `metric_value` member of the `MetricBundle` with the values returned by `run`.

(This is an over-simplification: the `MetricBundleGroup`'s `run_all` method has various optimizations and other complications that are not important here.)

## 3. Exploring existing metrics

MAF already contains a variety of metrics: before writing a new metric, it is worthwhile to check whether an existing one will do the job. A list of existing metrics with links to documentation can be found in the [MAF documentation](https://rubin-sim.lsst.io/maf-metric-list.html). 

A plain list of metrics can also be listed in a notebook with `maf.BaseMetric.help(doc=False)`.

If an existing metric does something close to what you want, you can find the source code for guidance thus:

In [ ]:
%pinfo maf.SumMetric

or dump the whole source code into a jupyter notebook:

In [ ]:
%psource maf.SumMetric

## 4. An exploratory metric

To get a feel for how metrics works, lets create a "dummy" metric that prints information on what arguments it was passed.

In [ ]:
import numpy as np


class Test1Metric(maf.BaseMetric):
    def __init__(self):
        # Set the columns we want from the database
        super().__init__(col=["fieldRA", "fieldDec", "HA"], metric_name="test1")

    def run(self, data_slice, slice_point=None):
        print("--- run ----")
        print(
            f"data_slice is a {type(data_slice)} of dtype {data_slice.dtype} with a shape of {data_slice.shape}"
        )
        print(f"slice_point is a {type(slice_point)} with value: {slice_point}")

        # Calculate the metric value here, using any values
        # you like from the dataSlice and slicePoint arguments.
        metric_value_for_this_slice = np.random.rand()
        return metric_value_for_this_slice

To apply our metric, we need to create a slicer with which to slice the data. Let's use the `maf.OneDSlicer`, which slices the data into bins according to some value. In this case, we divide the visits into large bins by airmass:

In [ ]:
slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.5)

Now, instantiate our metric, combine it with our slicer into a `maf.MetricBundle`, and compute the metrics in that bundle with a `maf.MetricBundleGroup`:

In [ ]:
metric = Test1Metric()
bundle = maf.MetricBundle(metric, slicer, "", run_name="sample_run")
bgroup = maf.MetricBundleGroup([bundle], opsim_db_fname, out_dir=output_dir)
bgroup.run_all()

Note that the `data_slice` `recarray` contained not only the fields we asked for, but two others as well: `airmass` and `observationStartLST`. These are present because the values were needed by the slicer, and by the stacker used to compute the `HA`.

We can look at the values calculated (or in this dummy example, randomly generated) with the `bundle` object:

In [ ]:
bundle.metric_values

In this example, the `run` method had access to the `slice_point` dictionary which included the `bin_left` field showing which bin the slice was. The metric could, therefore, have used it in its calculations. However, doing such would limit the slicers on which the metric could be run to only those slicers with a `bin_left` element in the `slice_point`.

## 5. A Workbook for developing metrics

Development of metrics usually involves repeated execution of successive modifications of metric code. The [New Metric Workbook](../science/New%20Metric%20Workbook.ipynb) (or in the github repo at https://github.com/lsst/rubin_sim_notebooks/blob/main/maf/science/NewMetricWorkbook.ipynb)  provides a template for development of metrics debugged through repeated execution within a jupyter notebook.

## 6. Applying a metric to the whole set of visits

Even if we do not want to slice the data, but instead want our `run` method to be passed the entire set of visits, we still need to supply a slicer to satisfy the MAF machinery. The `rubin_sim.maf.slicers.UniSlicer` slicer fills this role, creating one slice with all visits.

In [ ]:
slicer = maf.UniSlicer()
bundle = maf.MetricBundle(metric, slicer, "", run_name="unslicer_sample_run")
bgroup = maf.MetricBundleGroup([bundle], opsim_db_fname, out_dir=output_dir)
bgroup.run_all()

This time, `airmass` was not required to run the slicer, and it was not one of the fields listed in the `colNameArray` member of our metric, so it was not included in the `dataSlice` `recarray` passed to our metric.

## 7. General metrics

Some metrics are general in their applicability. For example, `maf.MedianMetric` can be applied to any numeric column, and does not use the contents of `slicePoint`s. In this case and those like it, the "work" of the `metric` object can mostly be handled by methods inhereted from `rubin_sim.maf.metrics.BaseMetric`.

Consider the definition of `maf.MedianMetric`:

In [ ]:
%psource maf.MedianMetric

How MAF know what column to get the median of? How does `self.colname` get set? 

The `maf.BaseMetric`'s `__init__` method has a `col` argument, and if it is set to a single string rather than a list of strings, the `colname` member takes the value of the one column name. So, the `maf.MedianMetric` must be initialized with the column name, and the `__init__` method of its parent class takes care of assigning the necessary members:

In [ ]:
slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.5)
metric = maf.MedianMetric(col="moonAlt")
bundle = maf.MetricBundle(metric, slicer, "")
bgroup = maf.MetricBundleGroup([bundle], opsim_db_fname, out_dir=output_dir)
bgroup.run_all()

Now we can look at the median moon altitudes for each of our large airmass bins:

In [ ]:
bundle.metric_values

We can also get the slice points used:

In [ ]:
bundle.slicer.slice_points

Or look in more detail at what is provided in a given slice_point (which may be a little different). 

In [ ]:
bundle.slicer[0]

## 8. Metric values

A metric's `run` method is called once on each slice generated by the slicer. Although it is usually a numeric value like an `int` or a `float`, it can potentially be any python object.

For example:

In [ ]:
class MyBespokeMoon:
    def __init__(self, alt, az):
        self.alt = alt
        self.az = az

    def __repr__(self):
        return f"Moon alt: {self.alt}, az: {self.az}"


class MyWeirdMetric(maf.BaseMetric):
    def __init__(self):
        super().__init__(col=["moonAlt", "moonAz"], metric_name="weirdness", metric_dtype="object")

    def run(self, data_slice, slice_point=None):
        if len(data_slice) is None:
            this_slice_value = None
        else:
            this_slice_value = MyBespokeMoon(data_slice[0].moonAlt, data_slice[0].moonAz)
        return this_slice_value

When we run this metric, we will get an array of our bespoke object back:

In [ ]:
slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.5)
metric = MyWeirdMetric()
bundle = maf.MetricBundle(metric, slicer, "")
bgroup = maf.MetricBundleGroup([bundle], opsim_db_fname, out_dir=output_dir)
bgroup.run_all()
bundle.metric_values

Note that the `metric_dtype` argument to the `maf.BaseMetric` initializer declared that the return type is an object.

Returning objects that are not of type `int` or `float` has a serious drawback: the visualization tools in MAF can generally only work with numbers:

In [ ]:
bgroup.plot_all(closefigs=False)

## 9. Turning complex metrics into scalars with "reduce" functions

If you have a metric which creates some arbitrary object (as `MyWeirdMetric` above), but one or more interesting scalars can be derived from it, then it can be convenient to creat `reduce` functions to provide access to these scalars. Doing this will prompt MAF to create plots of your scalar values, as well as run summary metrics on each of them (if summary metrics are configured on the metric bundle).   

In [ ]:
class MyWeirdMetric2(maf.BaseMetric):
    def __init__(self):
        super().__init__(col=["moonAlt", "moonAz"], metric_name="weirdness2", metric_dtype="object")

    def run(self, data_slice, slice_point=None):
        if len(data_slice) is None:
            this_slice_value = None
        else:
            this_slice_value = MyBespokeMoon(data_slice[0].moonAlt, data_slice[0].moonAz)
        return this_slice_value

    def reduce_alt(self, metric_value):
        return metric_value.alt

    def reduce_az(self, metric_value):
        return metric_value.az

In [ ]:
slicer = maf.OneDSlicer(slice_col_name="airmass", bin_min=1.0, bin_max=2.5, bin_size=0.1)
metric = MyWeirdMetric2()
bundle = maf.MetricBundle(metric, slicer, "")
bgroup = maf.MetricBundleGroup([bundle], opsim_db_fname, out_dir=output_dir)
bgroup.run_all()
bgroup.plot_all(closefigs=False)

With the addition of reduce methods (those beginning in `reduce`) that transform our custom object into scalars, plots that use scalars now appear. 

Full metric bundles corresponding to these new "reduced metrics" are present in the metric bundle group, and this provides access values themselves.

In [ ]:
bgroup.bundle_dict.keys()

In [ ]:
bgroup.bundle_dict["weirdness2_alt"].metric_values

## 10. Metrics for use with spatial slicers

If the metric can get all the data it needs from the columns that can be passed in as part of the `data_slice`, nothing special is required to use them to make a map on the sky.

Some metrics require data on the coordinates of the slice point, and only make sense when appied to a slicer than slices based on position in the sky.
MAF's `rubin_sim.maf.metrics.HealpixSlicer` is an example of such a slicer.
The `HealpixSlicer` creates `slice_point`s with keys that provide coordinate information on the slice:
- `sid`, the `healpix` id
- `ra`, the RA in radians
- `dec`, the declination in radians
- `nside`, the nside of the healpix pixelization

When combined with values that can be passed in with the `data_slice`, these values should provide the data needed to, for example, determine which part of the focal plane falls in the slice, or the area covered by the slice.

The `maf.ExgalM5` metric is **one example of a metric that requires such coordinate information for from the slicers** it can be used with:

In [ ]:
%psource maf.ExgalM5

- This metric demonstrates an additional feature of MAF metrics: the use of maps.
In this case, calculation of **the metric requires a dust map**, and this is specified in the call to the initializer of the parent class (`super().__init__...`).

- When the metric is evaluated, the dictionary supplied in the `slice_point` argument to `run` includes not only those generated by the slicer, but also `ebv`, the extinction from the dust map.

- There are other MAF `map`s available as well, each of which supplements the `slice_point` dictionary with different values:

In [ ]:
maf.BaseMap.help(doc=True)

## 11. What columns and stackers can I use?

You can get a list of columns available in an `opsim` output database by looking at the sqlite database output itself, or looking up the relevant documentation on the `rubin_scheduler` documentation pages (https://rubin-scheduler.lsst.io/fbs-output-schema.html).

This code will print a list of available stackers, and what columns they supply:

In [ ]:
import pprint

pprint.pprint({k: v.cols_added for k, v in maf.BaseStacker.registry.items()})

**Acknowledgements:** These tutorial notebooks have benefited from previous work in MAF tutorials, including not only the [tutorial notebooks in sims_MAF-contrib](https://github.com/LSST-nonproject/sims_maf_contrib/tree/master/tutorials) but also those by  Weixiang Yu, Gordon Richards, and Will Clarkson in their [LSST_OpSim](https://github.com/RichardsGroup/LSST_OpSim) repository, inspired the material to be included here. Stylistic elements of these notebooks were guided by the DP0.1 notebooks developed by Melissa Graham and the Rubin Observatory Community Engagement Team.